# PlanAgent 工作流程测试

测试流程：
1. 初始化基础设施（EventBus、ToolEventFactory、LLM Client）
2. 创建多个 Executor Agent（写作者、审阅者等），注册工具
3. 构建 PlanAgent 的 ContextEngine（含 ExecutorStatusProvider）
4. 创建 PlanAgent 实例，注入 executors
5. 提交任务，观察 plan 生成 → 步骤分发 → executor 执行 → 结果汇总 完整链路

In [1]:
import asyncio
import os
from dotenv import load_dotenv

load_dotenv()

True

## 1. 初始化基础设施

In [2]:
from infra.eventbus import EventBus
from domain.event import ToolEventFactory
from infra.tool.builtin.declare import *
from domain.agent.write.tools import *
from infra.LLM.LLM_infra import LLM_Client, LLM_Model_Provider

# 事件总线
bus = EventBus()
# 工具工厂
factory = ToolEventFactory(prefix="infra")._build()._resigister_bus(bus)
# LLM 客户端
llm_client = LLM_Client(
    url=os.getenv("LLM_BASE_URL", "https://api.minimaxi.com/v1"),
    model_class="MiniMax-M2.7",
    model_provider=LLM_Model_Provider.MINMAX,
    max_tokens=131072,
)

## 2. 注册工具 & 事件处理器

In [3]:
# 导入即注册：bash 工具 + 通用 succeeded/failed 处理器
from infra.tool.builtin.system import BASH, SystemTool
from infra.tool.tools_attach_methods import *

## 3. 创建 Executor Agent

PlanAgent 通过 `executors: dict[str, AgentBase]` 管理多个子 Agent。
这里创建两个简单的 ReACT executor：
- `writer` — 负责内容创作
- `operator` — 负责系统操作（bash）

In [4]:
from domain.agent_base import AgentBase
from domain.context.context import ContextEngine
from domain.context.providers import (
    UserPromptProvider,
    StateProvider,
    AvailableToolsProvider,
    HistoryProvider,
    ToolOutputProvider,
)
from domain.context.strategy import FullHistoryStrategy, RecencyStrategy
from domain.memory.short.default_short_term_memory import DefaultShortTermMemory


class WriterExecutor(AgentBase):
    """负责内容创作步骤的 ReACT executor"""

    def _build_agent_prompt(self) -> str:
        return f"""
你是一个内容创作执行者，当前工作目录为：{self.work_path}
你可以使用写作工具（需求分析、大纲生成、初稿写作等）和 bash 工具来完成任务。

## 输出格式
用 JSON 严格按以下格式回复：
{{
  "think": "你的思考过程",
  "tool_calls": [
    {{
      "tool_name": "工具名",
      "arguments": {{"参数名": "参数值"}},
      "reasoning": "为什么调用这个工具"
    }}
  ],
  "is_finished": false
}}

## 任务完成时输出
{{
  "think": "...",
  "tool_calls": [],
  "is_finished": true,
  "finish_reason": "完成原因"
}}
"""


class OperatorExecutor(AgentBase):
    """负责系统操作步骤的 ReACT executor"""

    def _build_agent_prompt(self) -> str:
        return f"""
你是一个系统操作执行者，当前工作目录为：{self.work_path}
你可以使用 bash 工具执行命令来完成任务。

## 输出格式
用 JSON 严格按以下格式回复：
{{
  "think": "你的思考过程",
  "tool_calls": [
    {{
      "tool_name": "bash",
      "arguments": {{"command": "要执行的命令"}},
      "reasoning": "为什么执行这个命令"
    }}
  ],
  "is_finished": false
}}

## 任务完成时输出
{{
  "think": "...",
  "tool_calls": [],
  "is_finished": true,
  "finish_reason": "完成原因"
}}
"""

In [5]:
# writer executor：暴露 write_agent 组工具
writer_memory = DefaultShortTermMemory(["tool_respond", "agent_history"])
writer_context = ContextEngine(
    providers=[
        UserPromptProvider(),
        StateProvider(),
        AvailableToolsProvider(["write_agent","system"]),
        HistoryProvider(writer_memory, "agent_history", FullHistoryStrategy()),
        ToolOutputProvider(writer_memory, "tool_respond", FullHistoryStrategy() | RecencyStrategy(10)),
    ],
    memory=writer_memory,
)
writer = WriterExecutor(
    id="writer",
    name="内容创作执行者",
    llm=llm_client,
    context=writer_context,
)

# operator executor：暴露 system 组工具
operator_memory = DefaultShortTermMemory(["tool_respond", "agent_history"])
operator_context = ContextEngine(
    providers=[
        UserPromptProvider(),
        StateProvider(),
        AvailableToolsProvider(["system"]),
        HistoryProvider(operator_memory, "agent_history", FullHistoryStrategy()),
        ToolOutputProvider(operator_memory, "tool_respond", FullHistoryStrategy() | RecencyStrategy(10)),
    ],
    memory=operator_memory,
)
operator = OperatorExecutor(
    id="operator",
    name="系统操作执行者",
    llm=llm_client,
    context=operator_context,
)

print(f"Writer: id={writer.id}, tools=write_agent+system")
print(f"Operator: id={operator.id}, tools=system")

Writer: id=writer, tools=write_agent+system
Operator: id=operator, tools=system


## 4. 创建 PlanAgent

In [6]:
from domain.agent.plan.planAgent import PlanAgent
from domain.agent.plan.providers import ExecutorStatusProvider, PlanStepPromptProvider

# PlanAgent 的上下文引擎：
#   - UserPromptProvider：注入用户需求
#   - StateProvider：注入当前状态
#   - ExecutorStatusProvider：注入各 executor 状态，供 LLM 规划
plan_memory = DefaultShortTermMemory(["tool_respond", "agent_history"])
plan_context = ContextEngine(
    providers=[
        UserPromptProvider(),
        StateProvider(),
        ExecutorStatusProvider(),
    ],
    memory=plan_memory,
)

# executors 字典：key = executor_id，value = AgentBase 实例
# executor_id 会出现在 LLM 生成的计划步骤中，PlanAgent 据此分发
executors = {
    "writer": writer,
    "operator": operator,
}

plan_agent = PlanAgent(
    id="plan_agent_001",
    name="任务编排者",
    llm=llm_client,
    context=plan_context,
    executors=executors,
)

print(f"PlanAgent ID: {plan_agent.id}")
print(f"PlanAgent Name: {plan_agent.name}")
print(f"Executors: {list(executors.keys())}")

PlanAgent ID: plan_agent_001
PlanAgent Name: 任务编排者
Executors: ['writer', 'operator']


## 5. 完整运行测试

PlanAgent 工作流：
1. `start(prompt)` → 保存 prompt，初始化 executor 状态
2. `_generate_plan()` → ContextEngine 构建 prompt → LLM 生成计划 JSON → 解析为 Plan
3. `_execute_plan()` → 按 executor_id 分组步骤 → `asyncio.gather` 并行执行每组
4. `_run_executor_steps()` → 逐步执行：
   - 设置 `state["current_step"]`
   - PlanStepPromptProvider 构造步骤 prompt
   - 调用 `executor.start(step_prompt)` → ReACT 循环
   - 更新步骤状态（done/failed）
5. `_summarize_result()` → 汇总所有 executor 结果

In [7]:
async def run_plan_agent_test():
    """
    PlanAgent 工作流程完整测试

    预期流程：
    1. PlanAgent 接收用户需求
    2. LLM 生成结构化计划（含 executor_id 分配）
    3. 按 executor_id 分发步骤给对应 executor
    4. 每个 executor 独立执行 ReACT 循环
    5. PlanAgent 汇总所有结果
    """
    print("=" * 60)
    print("PlanAgent 工作流程测试")
    print("=" * 60)

    test_prompt = """请帮我完成以下任务：
1. 写一篇关于人工智能发展历程的短文，约500字
2. 将写好的内容保存为 ai_history.md 文件
完成后汇报结果。
"""

    print(f"\n测试任务:\n{test_prompt}")
    print("\n开始执行...\n")
    print("=" * 60)

    try:
        await plan_agent.start(prompt=test_prompt)

        print("\n" + "=" * 60)
        print("PlanAgent 执行完成！")
        print("=" * 60)

        # 查看计划状态
        plan_dict = plan_agent.states.get("plan", {})
        steps = plan_dict.get("steps", [])
        print(f"\n计划步骤数: {len(steps)}")
        for step in steps:
            print(f"  [{step.get('step_id')}] {step.get('title')} → status={step.get('status')}, executor={step.get('executor_id', 'N/A')}")

        # 查看 executor 状态
        executor_status = plan_agent.states.get("executors", {})
        print(f"\nExecutor 状态:")
        for eid, estatus in executor_status.items():
            print(f"  {eid}: finished={estatus.get('is_finished')}, reason={estatus.get('finish_reason', 'N/A')}")

        # 查看最终汇总
        print(f"\n最终汇总: {plan_agent.states.get('final', 'N/A')[:500]}")
        print(f"完成原因: {plan_agent.states.get('finish_reason', 'N/A')}")

        # 查看各 executor 的工具调用历史
        for eid, executor in executors.items():
            history = executor.states.get("tool_history", [])
            print(f"\n[{eid}] 工具调用历史: {' -> '.join(history) if history else '无'}")

        return plan_agent

    except Exception as e:
        print(f"\n执行出错: {str(e)}")
        import traceback
        traceback.print_exc()

In [8]:
# 完整运行测试
await run_plan_agent_test()

PlanAgent 工作流程测试

测试任务:
请帮我完成以下任务：
1. 写一篇关于人工智能发展历程的短文，约500字
2. 将写好的内容保存为 ai_history.md 文件
完成后汇报结果。


开始执行...



```json
{
  "steps": [
    {
      "step_id": "1",
      "title": "撰写人工智能发展历程短文",
      "detail": "由内容创作执行者撰写约500字的关于人工智能发展历程的短文，内容应涵盖AI从概念萌芽到现代发展的关键阶段。",
      "executor_id": "writer"
    },
    {
      "step_id": "2",
      "title": "保存文件",
      "detail": "将撰写好的内容保存为 ai_history.md 文件，确保文件编码正确且内容完整。",
      "executor_id": "operator"
    }
  ]
}
```

我将完成当前步骤：撰写人工智能发展历程短文并保存为文件。

首先让我分析用户需求，然后进行内容创作

```json
{
  "think": "任务是要保存关于人工智能发展历程的短文到 ai_history.md 文件中。当前是步骤2（保存文件）。我需要先检查当前目录状态，看是否：
{
  "think": "用户需要一篇约500字的关于人工智能发展历程的短文，需要涵盖从概念萌芽到现代发展的关键阶段。这是一个信息已有写好的内容，然后创建或更新 ai_history.md 文件。由于我无法直接获取之前的步骤1的内容，我需要根据用户需求自己撰写这篇性文章写作任务，需要清晰、准确地描述AI的历史发展脉络。我需要先分析需求，然后撰写初稿，最后保存为md文件。",
  "tool_calls": [
    {
约500字的文章并保存。",
  "tool_calls": [
    {
      "tool_name": "bash",
      "arguments": {"command": "ls -la /Users/zxcvbzzy1/Desktop/项目/agent_flow/temp/"},
      "reasoning": "先查看

## 6. 单独测试计划生成（不执行）

只看 LLM 能否正确生成带 executor_id 的计划 JSON

In [ ]:
async def test_plan_generation_only():
    """
    只测试计划生成环节，不执行。
    验证 LLM 是否能根据 ExecutorStatusProvider 输出正确的计划 JSON。
    """
    print("=" * 60)
    print("计划生成单独测试")
    print("=" * 60)

    # 模拟初始状态
    plan_agent.states["prompt"] = "写一篇500字的AI发展短文并保存为文件"
    plan_agent.states["executors"] = plan_agent._executor_status()

    # 调用 _generate_plan
    print("\n[1] 生成计划...")
    plan = await plan_agent._generate_plan()

    print(f"\n[2] 计划内容:")
    print(f"   步骤数: {len(plan.steps)}")
    print(f"   是否完成: {plan.finished}")
    for step in plan.steps:
        print(f"   [{step.step_id}] {step.title}")
        print(f"       detail: {step.detail}")
        print(f"       status: {step.status}")

    print("\n[3] Plan dict:")
    print(plan.to_dict())

    return plan

In [ ]:
await test_plan_generation_only()

## 7. 单独测试 Executor 分发逻辑

手动构造 Plan，测试步骤按 executor_id 分组分发

In [ ]:
from domain.state import Plan, PlanStep

def test_step_grouping():
    """
    手动构造 Plan，验证步骤按 executor_id 分组的逻辑。
    不涉及 LLM 调用。
    """
    print("=" * 60)
    print("步骤分组逻辑测试")
    print("=" * 60)

    plan = Plan()
    plan.add_steps([
        {"step_id": "1", "title": "需求分析", "detail": "分析AI发展短文需求"},
        {"step_id": "2", "title": "撰写短文", "detail": "写500字AI发展历程"},
        {"step_id": "3", "title": "保存文件", "detail": "将内容保存为md文件"},
        {"step_id": "4", "title": "验证文件", "detail": "确认文件已保存"},
    ])

    # 手动给步骤分配 executor_id（模拟 LLM 输出）
    plan.steps[0].executor_id = "writer"   # 需求分析 → writer
    plan.steps[1].executor_id = "writer"   # 撰写短文 → writer
    plan.steps[2].executor_id = "operator" # 保存文件 → operator
    plan.steps[3].executor_id = "operator" # 验证文件 → operator

    # 按 executor_id 分组
    from collections import defaultdict
    grouped = defaultdict(list)
    for step in plan.steps:
        grouped[step.executor_id].append(step)

    print(f"\n分组结果:")
    for eid, steps in grouped.items():
        step_names = [f"[{s.step_id}]{s.title}" for s in steps]
        print(f"  {eid}: {' → '.join(step_names)}")

    # 验证：未知 executor_id 的步骤会被标记为 failed
    plan2 = Plan()
    plan2.add_steps([
        {"step_id": "1", "title": "任务A", "detail": "由未知执行者处理"},
    ])
    plan2.steps[0].executor_id = "unknown_executor"

    for step in plan2.steps:
        if step.executor_id not in executors:
            step.status = "failed"
            step.note = f"未知 executor_id: {step.executor_id}"

    print(f"\n未知 executor 测试:")
    print(f"  step[{plan2.steps[0].step_id}] status={plan2.steps[0].status}, note={plan2.steps[0].note}")

In [ ]:
test_step_grouping()

## 8. 单独测试 PlanStepPromptProvider

验证发送给 executor 的步骤 prompt 格式

In [ ]:
def test_step_prompt_provider():
    """
    测试 PlanStepPromptProvider 的输出格式。
    验证 executor 收到的 prompt 是否包含当前步骤信息。
    """
    print("=" * 60)
    print("PlanStepPromptProvider 测试")
    print("=" * 60)

    provider = PlanStepPromptProvider()

    # 模拟 PlanAgent 设置 current_step 后的状态
    mock_state = {
        "prompt": "写一篇关于人工智能发展历程的短文并保存为文件",
        "current_step": {
            "step_id": "2",
            "title": "撰写短文",
            "detail": "写500字AI发展历程，涵盖从图灵测试到大模型时代",
        },
    }

    result = provider.get(mock_state)
    print(f"\nProvider 输出:")
    for item in result:
        print(item)

    # 测试无 current_step 时
    empty_result = provider.get({"prompt": "测试"})
    print(f"\n无 current_step 时: {empty_result}")

In [ ]:
test_step_prompt_provider()

## 9. 单独测试 ExecutorStatusProvider

验证 executor 状态信息是否正确注入到 PlanAgent 的 prompt

In [ ]:
def test_executor_status_provider():
    """
    测试 ExecutorStatusProvider 的输出。
    验证 LLM 规划时能看到各 executor 的可用状态。
    """
    print("=" * 60)
    print("ExecutorStatusProvider 测试")
    print("=" * 60)

    provider = ExecutorStatusProvider()

    # 初始状态：所有 executor 可用
    state_initial = {"executors": plan_agent._executor_status()}
    result = provider.get(state_initial)
    print(f"\n[初始状态]")
    for item in result:
        print(item)

    # 模拟 writer 已完成一个任务
    writer.states["is_finished"] = True
    writer.states["finish_reason"] = "需求分析完成"
    state_after = {"executors": plan_agent._executor_status()}
    result_after = provider.get(state_after)
    print(f"\n[writer 完成后]")
    for item in result_after:
        print(item)

    # 重置
    writer.states["is_finished"] = False
    writer.states["finish_reason"] = ""

In [ ]:
test_executor_status_provider()